# P5-4 KRIT 공고 + P2-6 열린재정 예산 정제 → `clean_` 3표 적재

작성 2026-09-19 (담당분배 명세 `docs/reference/clean-conversion-spec-2026-09-18.md` §1 공통 · §3 6번(openfiscal) · §6 4번(krit_task)).
규칙 근거는 `docs/reference/data-cleaning-rules.md` §1·§2-4(B1 KRIT)·§2-12(A9 열린재정)·§3·§4,
DDL 은 `db/alter_2026-09-19_krit_budget_clean.sql`(→ `db/schema.sql` 반영).

| 원본 파일(데이터셋 키) | 정제 결과(clean_) | 기대 행 수 | 비고 |
|---|---|---|---|
| `raw_krit_task` 96 | `clean_krit_task` — 차수×공고유형×과제번호(재부여) | 96 (제외 0) | `is_latest=1` 73 기대(차수별 과제 수) |
| `raw_openfiscal_program_budget` 2,860 | `clean_openfiscal_program_budget` — raw 1:1 | 2,860 (제외 0) | 천원 → 억원 열 추가, 2027 `is_unconfirmed=1` |
| 〃 (파생) | `clean_openfiscal_program_link` — 세부사업명 개편 연결표 | 표기변경(확정) + 국방기술개발 승계 후보 | 근거 없는 1:1 연결은 만들지 않는다 |

원칙
- 원본은 `data/raw/` 파일을 `read_raw()`로 읽는다(RDS `raw_` 표는 2026-09-22 삭제 — `raw_row_id` = 파서 순번, 파일명 정렬 × 행 순). 대상 `clean_`이 **비어 있을 때만** 적재한다(재적재는 admin `TRUNCATE` 후 재실행).
  제외한 raw 행은 `clean_excluded_row`에 사유와 함께 남겨 `raw = clean + excluded`를 검산한다(이번 2표는 제외 0 기대).
- **HS6 품목군 대응 열(`category`·`hs6`·`category_link_status`)은 없다** — 2026-09-21 카테고리 맵 폐기로 삭제
  (`CLAUDE.md` 핵심 설계 제약. 과제명 키워드로 HS6를 붙이지 않는다).
  예산 `budget_group_candidate`는 **키워드 후보**일 뿐 확정 분류가 아니다.
- 금액 단위는 열 이름에 적는다: `_krw_k` = 천원, `_100m_krw` = 억원(= 천원 ÷ 100,000). 원문 텍스트는 `gov_fund_text`에 보존.
- 담당자명·연락처 열은 clean 에 넣지 않는다(두 raw 에는 애초에 없다).
- 단계별 건수는 `meta_load_log`에 기록한다. 접속은 `scripts/load_db.py`의 `connect()`(→ `.env`, `etl_rw`).

In [1]:
import sys, re, json
from pathlib import Path
from datetime import datetime
from decimal import Decimal
import pandas as pd
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user, read_raw, log_raw_stage   # .env 읽기 · meta_load_log 기록 · 원본 파일 읽기
RAW = {}   # 원본 파일 DataFrame 캐시 {데이터셋 키: read_raw 결과}. RDS raw_ 표는 2026-09-22 삭제 — row_id = 파서 순번 = clean.raw_row_id
def raw(table, cols=None):
    if table not in RAW:
        RAW[table] = read_raw(table)
    df = RAW[table]
    return (df[["row_id"] + [c for c in cols if c != "row_id"]] if cols else df).copy()

NB_TAG = "notebooks/05_clean_krit_budget.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
NOW = datetime.now().replace(microsecond=0)
def q(sql, params=None):
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}"); return cur.fetchone()[0]
def py(v):
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT or (isinstance(v, pd._libs.missing.NAType)): return None
    if isinstance(v, Decimal): return v
    if hasattr(v, "item") and not isinstance(v, str): return v.item()
    if isinstance(v, pd.Timestamp): return v.to_pydatetime().date()
    return v
def insert_df(table, df, cols, chunk=2000):
    rows = [tuple(py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO {table} ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    for i in range(0, len(rows), chunk):
        cur.executemany(sql, rows[i:i+chunk])
    return len(rows)

excluded = []   # (table_name, raw_row_id, reason_code, note)
print("connected as", USER, "| pandas", pd.__version__)
print("대상 clean 현재 행수:", {t: count(t) for t in
      ("clean_krit_task", "clean_openfiscal_program_budget", "clean_openfiscal_program_link")})

connected as etl_rw@% | pandas 3.0.5
대상 clean 현재 행수: {'clean_krit_task': 96, 'clean_openfiscal_program_budget': 2860, 'clean_openfiscal_program_link': 22}


## 1. KRIT 부품국산화 공고 → `clean_krit_task`

- **키 재부여**: raw `task_seq`('순')는 표(`구분(표제목)` = 핵심부품/수출연계/전략부품)마다 1부터 다시 시작한다.
  23-4차 본공고 1~16 + 1~2, 25-1차 수정 1~18 + 1~4 처럼 겹치고, 24-1차 예비 11행은 '순' 자리에 구분값(`핵심`·`수출`)이 들어 있다.
  → PK `task_no`는 (`round_id`, `notice_type`) 안에서 raw `row_id` 순으로 **새로 매기고**, 원문은 `task_seq_text`에 남긴다.
- `round_label`('26-1차') → `round_id`('26-1') · `round_year`(2026) · `round_seq`(1). `notice_type`은 raw 그대로(NULL → `미확인`).
- `program_type`은 `extra_json["구분(표제목)"]`, 없으면 `task_seq`의 구분값(`핵심`→핵심부품, `수출`→수출연계). 둘 다 없으면 NULL(26-1차 2행).
- **금액 단위가 차수마다 다르다**: `14.64억`(23-4차) / `10.35억원`(26-2차 본공고) / `1,657`(26-2차 예비 — 원본 헤더가 `최대 정부\n지원금(백만원)`,
  `extra_json["원본열명"]`에서 확인) / 24-1차 예비 11행은 정부지원금 자체가 없다(총과제비만 `extra_json`, **다른 지표라 옮기지 않는다**).
  → `gov_fund_100m_krw`(억원) = 억·억원이면 그대로, 백만원이면 ÷100. 단위 근거는 `gov_fund_unit_text`, 원문은 `gov_fund_text`.
- **`is_latest`**: 같은 차수에서 예비→본공고, 수정→재공고로 같은 과제가 다시 공고된다(25-1차 3쌍, 26-2차 20쌍 — 과제명 완전 일치).
  (`round_id`, `task_name`)별로 `notice_order`(예비1 < 본공고2 < 수정3 < 재공고4, **잠정 규칙**)가 가장 큰 행 1개만 `is_latest=1`.
  기존 열 `is_counted`는 같은 값으로 채운다. 차수별 과제 수는 이 열로만 세고 공고 행을 더하지 않는다.
- 과제명 키워드로 HS6를 매핑하지 않는다 — HS6 대응 열은 2026-09-21 카테고리 맵 폐기로 삭제됐다.
- `extra_json`은 clean 에 복사하지 않는다(명세 "그대로 둠"). 필요하면 `raw_row_id`로 raw 를 본다.

In [2]:
rk = raw("raw_krit_task", ["round_label", "notice_type", "task_seq", "task_name", "gov_fund_text", "dev_period_text", "note", "extra_json", "source_file"])
print("원본 전체:", len(rk), "| 차수:", rk["round_label"].nunique(), "| 공고유형:", rk["notice_type"].value_counts(dropna=False).to_dict())

k = rk.copy()
k["round_id"] = k["round_label"].astype(str).str.replace("차$", "", regex=True)
assert k["round_id"].str.fullmatch(r"\d{2}-\d{1,2}").all(), "round_label 형식 예외"
k["round_year"] = 2000 + k["round_id"].str.split("-").str[0].astype(int)
k["round_seq"] = k["round_id"].str.split("-").str[1].astype(int)
k["notice_type"] = k["notice_type"].fillna("미확인").replace({"": "미확인"})
NOTICE_ORDER = {"미확인": 0, "예비": 1, "본공고": 2, "수정": 3, "재공고": 4}
assert set(k["notice_type"]) <= set(NOTICE_ORDER), set(k["notice_type"]) - set(NOTICE_ORDER)
k["notice_order"] = k["notice_type"].map(NOTICE_ORDER).astype(int)

def jload(s):
    if not isinstance(s, str) or not s.strip():
        return {}
    try:
        return json.loads(s)
    except Exception:
        return {}
ej = k["extra_json"].map(jload)
k["task_seq_text"] = k["task_seq"].astype("object").where(k["task_seq"].notna(), None)
SEQ_AS_TYPE = {"핵심": "핵심부품", "수출": "수출연계"}
k["program_type"] = ej.map(lambda d: d.get("구분(표제목)")).fillna(k["task_seq_text"].map(SEQ_AS_TYPE))
print("program_type:", k["program_type"].value_counts(dropna=False).to_dict())

# 키 재부여 (raw row_id 순)
k = k.sort_values("row_id")
k["task_no"] = k.groupby(["round_id", "notice_type"])["row_id"].rank(method="first").astype(int)
assert not k.duplicated(["round_id", "notice_type", "task_no"]).any(), "PK 중복"
dup_orig = k.duplicated(["round_id", "notice_type", "task_seq_text"]).sum()
print("원문 순번을 그대로 PK로 썼다면 충돌했을 행:", int(dup_orig), "| 재부여 후 PK 고유:",
      k[["round_id", "notice_type", "task_no"]].drop_duplicates().shape[0])

# 금액: 단위가 차수마다 다르다
orig_hdr = ej.map(lambda d: (d.get("원본열명") or {}).get("gov_fund_text") or "")
txt = k["gov_fund_text"].fillna("").astype(str).str.strip()
unit = pd.Series([None] * len(k), index=k.index, dtype="object")
unit[txt == ""] = "없음"
unit[(txt != "") & txt.str.endswith("억원")] = "억원"
unit[(txt != "") & txt.str.endswith("억")] = "억"
unit[(unit.isna()) & orig_hdr.str.contains("백만원")] = "백만원"
k["gov_fund_unit_text"] = unit
assert k["gov_fund_unit_text"].notna().all(), txt[k["gov_fund_unit_text"].isna()].tolist()
num = pd.to_numeric(txt.str.replace(r"[^\d.]", "", regex=True).replace({"": None}), errors="coerce")
k["gov_fund_100m_krw"] = np.where(k["gov_fund_unit_text"] == "백만원", num / 100, num)
k.loc[k["gov_fund_unit_text"] == "없음", "gov_fund_100m_krw"] = np.nan
k["gov_fund_100m_krw"] = k["gov_fund_100m_krw"].round(2)
print("금액 단위:", k["gov_fund_unit_text"].value_counts().to_dict(),
      "| 억원 환산 NULL:", int(k["gov_fund_100m_krw"].isna().sum()))

k["dev_period_months"] = pd.to_numeric(
    k["dev_period_text"].fillna("").astype(str).str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")
k["note"] = k["note"].fillna("").astype(str).str.strip().replace({"": None, "-": None, "·": None, "*": None})
k["task_name"] = k["task_name"].astype(str).str.strip()
assert k["task_name"].ne("").all() and k["dev_period_months"].notna().all()

# is_latest: (차수, 과제명)별 최신 공고 1행
k["dup_task_row_count"] = k.groupby(["round_id", "task_name"])["row_id"].transform("size").astype(int)
k = k.sort_values(["round_id", "task_name", "notice_order", "row_id"])
k["is_latest"] = (k.groupby(["round_id", "task_name"]).cumcount(ascending=False) == 0).astype(int)
k = k.sort_values("row_id")
k["is_counted"] = k["is_latest"]
k["raw_row_id"] = k["row_id"]
k["cleaned_at"] = NOW
k["cleaned_by"] = USER

print("중복 처리 후:", len(k), "(raw 1:1, 제외 0)")
print("관련 후보 — is_latest=1:", int(k["is_latest"].sum()),
      "| 차수별:", k.groupby("round_id")["is_latest"].agg(["sum", "size"]).to_dict("index"))
print("중복 공고 행(dup_task_row_count>1):", int((k["dup_task_row_count"] > 1).sum()),
      "| 차수×공고유형:", k.groupby(["round_id", "notice_type"]).size().to_dict())
print("정부지원금 합(억원, is_latest=1만):", round(float(k.loc[k["is_latest"] == 1, "gov_fund_100m_krw"].sum()), 2),
      "| 개발기간 개월 min/max:", int(k["dev_period_months"].min()), int(k["dev_period_months"].max()))

원본 전체: 96 | 차수: 5 | 공고유형: {'본공고': 40, '예비': 31, '수정': 22, '재공고': 3}
program_type: {'핵심부품': 72, '전략부품': 19, '수출연계': 3, nan: 2}
원문 순번을 그대로 PK로 썼다면 충돌했을 행: 30 | 재부여 후 PK 고유: 96
금액 단위: {'억': 45, '억원': 20, '백만원': 20, '없음': 11} | 억원 환산 NULL: 11
중복 처리 후: 96 (raw 1:1, 제외 0)
관련 후보 — is_latest=1: 73 | 차수별: {'23-4': {'sum': 18, 'size': 18}, '24-1': {'sum': 11, 'size': 11}, '25-1': {'sum': 22, 'size': 25}, '26-1': {'sum': 2, 'size': 2}, '26-2': {'sum': 20, 'size': 40}}
중복 공고 행(dup_task_row_count>1): 46 | 차수×공고유형: {('23-4', '본공고'): 18, ('24-1', '예비'): 11, ('25-1', '수정'): 22, ('25-1', '재공고'): 3, ('26-1', '본공고'): 2, ('26-2', '본공고'): 20, ('26-2', '예비'): 20}
정부지원금 합(억원, is_latest=1만): 3732.07 | 개발기간 개월 min/max: 18 60


## 2. 열린재정 세부사업 예산 → `clean_openfiscal_program_budget`

- raw 2,860행 **1:1**(제외 0). 업무 키 (`회계연도`,`프로그램명`,`단위사업명`,`세부사업명`) 중복 0 = 2,860(실측) — PK 는 `raw_row_id`.
- **빼는 열**(명세 §3-4 · `data-cleaning-rules` §2-12): `No.`·`소관명`(방위사업청 1값)·`회계명`(일반회계 1값)·`분야명`·`부문명`(각 1값)
  그리고 **`계정명`(`sub_account_name`) — 2,860행 전부 공란**이라 clean 에 두지 않는다(아래 셀에서 실측 확인).
- 금액: `8,080,000` 같은 쉼표 문자열 → `gov_plan_krw_k`·`confirmed_krw_k`(천원, BIGINT) + `*_100m_krw`(억원 = 천원 ÷ 100,000, `Decimal` 나눗셈만 — 반올림하지 않는다).
- **`is_unconfirmed`**: 그 회계연도의 국회확정금액 합이 0이면 1(= 2027 268행, 정부안만 있는 상태). **0원이 아니라 미확정**이다.
  연도 합이 0이 아닌 해의 개별 0행은 실제 미반영이므로 0 그대로 둔다. `amount_basis`(확정/정부안)는 `v_budget_rnd_yearly`와 같은 규칙.
- `sub_program_key` = 세부사업명에서 공백·`(R&D)`·`(방사청)` 접미를 뺀 표기 정규화 키. `sub_program_first_year`/`last_year`로 개편 구간을 표시한다.
- **`budget_group_candidate`(⑤ 탭 예산 3선)** — 키워드 **후보**일 뿐 확정 분류가 아니다.
  우선순위: 세부사업명에 `반도체` → `국방반도체`, `부품국산화` → `부품국산화`, 그 밖에 단위사업이 `국방기술개발`이면 `국방기술개발`.
  앞의 둘도 `국방기술개발` 단위사업 안에 있으므로(`is_unit_tech_dev=1`) **세 값을 더하면 중복 집계된다**.

In [3]:
rb = raw("raw_openfiscal_program_budget", ["fiscal_year", "ministry_name", "account_name", "sub_account_name", "sector_name", "field_name", "program_name", "unit_program_name", "sub_program_name", "expense_type", "outlay_type", "gov_plan_krw_k", "confirmed_krw_k"])
print("원본 전체:", len(rb))
for c in ("ministry_name", "account_name", "sector_name", "field_name", "expense_type", "outlay_type"):
    print(f"  {c}: {rb[c].nunique()}종 {sorted(rb[c].dropna().unique())[:4]}")
blank_sub = (rb["sub_account_name"].fillna("").astype(str).str.strip() == "").sum()
print("계정명(sub_account_name) 공란:", int(blank_sub), "/", len(rb), "→ 전부 공란이면 clean 에서 제외:", blank_sub == len(rb))
assert blank_sub == len(rb), "계정명에 값이 있는 행이 생겼다 — 제외 규칙 재검토"

b = rb.copy()
b["fiscal_year"] = b["fiscal_year"].astype(int)
key = ["fiscal_year", "program_name", "unit_program_name", "sub_program_name"]
print("업무 키 고유:", b[key].drop_duplicates().shape[0], "/", len(b))
assert not b.duplicated(key).any(), "업무 키 중복 발생"

def to_krw_k(s):
    x = s.fillna("").astype(str).str.replace(",", "", regex=False).str.strip()
    assert x.str.fullmatch(r"-?\d+").all(), x[~x.str.fullmatch(r"-?\d+")].unique()[:5]
    return x.astype("int64")
b["gov_plan_krw_k"] = to_krw_k(b["gov_plan_krw_k"])
b["confirmed_krw_k"] = to_krw_k(b["confirmed_krw_k"])
b["gov_plan_100m_krw"] = [Decimal(int(v)) / Decimal(100000) for v in b["gov_plan_krw_k"]]
b["confirmed_100m_krw"] = [Decimal(int(v)) / Decimal(100000) for v in b["confirmed_krw_k"]]
print("음수 금액 행:", int((b["gov_plan_krw_k"] < 0).sum()), "/", int((b["confirmed_krw_k"] < 0).sum()))

year_conf = b.groupby("fiscal_year")["confirmed_krw_k"].sum()
unconf_years = sorted(year_conf[year_conf == 0].index.tolist())
b["is_unconfirmed"] = b["fiscal_year"].isin(unconf_years).astype(int)
b["amount_basis"] = np.where(b["is_unconfirmed"] == 1, "정부안", "확정")
print("확정액 합이 0인 회계연도(=정부안):", unconf_years, "| 해당 행:", int(b["is_unconfirmed"].sum()))
print("연도 합은 0이 아닌데 개별 확정액이 0인 행(실제 미반영, 0 유지):",
      int(((b["confirmed_krw_k"] == 0) & (b["is_unconfirmed"] == 0)).sum()))

NORM_TOKENS = ["(R&D)", "(방사청)"]
def skey(s):
    out = s.astype(str).str.replace(r"\s+", "", regex=True)
    for t in NORM_TOKENS:
        out = out.str.replace(t, "", regex=False)
    return out
b["sub_program_key"] = skey(b["sub_program_name"])
yr = b.groupby("sub_program_key")["fiscal_year"]
b["sub_program_first_year"] = yr.transform("min").astype(int)
b["sub_program_last_year"] = yr.transform("max").astype(int)
print("세부사업명 원문 고유:", b["sub_program_name"].nunique(), "→ 정규화 키 고유:", b["sub_program_key"].nunique())

b["is_unit_tech_dev"] = (b["unit_program_name"] == "국방기술개발").astype(int)
cond = [b["sub_program_name"].str.contains("반도체"),
        b["sub_program_name"].str.contains("부품국산화"),
        b["is_unit_tech_dev"] == 1]
b["budget_group_candidate"] = np.select(cond, ["국방반도체", "부품국산화", "국방기술개발"], default=None)
b["budget_group_basis"] = np.select(
    cond,
    ["세부사업명에 '반도체' 포함(키워드 후보)", "세부사업명에 '부품국산화' 포함(키워드 후보)",
     "단위사업명 = 국방기술개발(⑤ 탭 대표 시계열)"], default=None)
b["cleaned_at"] = NOW
b["cleaned_by"] = USER
b["raw_row_id"] = b["row_id"]
print("중복 처리 후:", len(b), "(raw 1:1, 제외 0)")
print("관련 후보 — budget_group_candidate:", b["budget_group_candidate"].value_counts(dropna=False).to_dict())
print("연도별 정부안 합(억원):",
      {int(y): float(round(v, 1)) for y, v in b.groupby("fiscal_year")["gov_plan_100m_krw"].sum().items()})
print("국방기술개발 단위사업 정부안 합(억원):",
      {int(y): float(round(v, 1)) for y, v in
       b[b["is_unit_tech_dev"] == 1].groupby("fiscal_year")["gov_plan_100m_krw"].sum().items()})

원본 전체: 2860
  ministry_name: 1종 ['방위사업청']
  account_name: 1종 ['일반회계']
  sector_name: 1종 ['국방']
  field_name: 1종 ['방위력개선']
  expense_type: 3종 ['기본경비(경상운영비)', '인건비(기금인건비)', '주요사업비(기금사업비)']
  outlay_type: 2종 ['내부거래지출(예산=>기금간거래)', '일반지출']
계정명(sub_account_name) 공란: 2860 / 2860 → 전부 공란이면 clean 에서 제외: True
업무 키 고유: 2860 / 2860
음수 금액 행: 0 / 0
확정액 합이 0인 회계연도(=정부안): [2027] | 해당 행: 268
연도 합은 0이 아닌데 개별 확정액이 0인 행(실제 미반영, 0 유지): 10
세부사업명 원문 고유: 674 → 정규화 키 고유: 606
중복 처리 후: 2860 (raw 1:1, 제외 0)
관련 후보 — budget_group_candidate: {nan: 2767, '국방기술개발': 85, '부품국산화': 7, '국방반도체': 1}
연도별 정부안 합(억원): {2016: 116803.3, 2017: 121589.5, 2018: 134825.5, 2019: 153733.2, 2020: 166915.4, 2021: 170737.8, 2022: 173364.8, 2023: 170179.0, 2024: 177985.7, 2025: 180712.5, 2026: 201743.9, 2027: 227224.2}
국방기술개발 단위사업 정부안 합(억원): {2016: 8163.5, 2017: 8135.8, 2018: 9107.8, 2019: 9487.8, 2020: 10053.3, 2021: 13877.9, 2022: 21739.6, 2023: 27174.6, 2024: 23738.7, 2025: 25371.1, 2026: 28397.7, 2027: 30741.4}


## 3. 세부사업명 개편 연결표 → `clean_openfiscal_program_link`

명세 §3-4 "사업명 개편 구간은 상위 사업(단위사업) 합계로 연결 — 연결표를 따로 남김". 공식 개편 고시를 확인한 것이 아니므로 **근거 없는 1:1 연결을 만들지 않는다**.

- 연결을 찾는 범위는 **`단위사업명`(`unit_program_name`)**이다. `프로그램명`은 2018년에 `국방연구개발사업` → `방위사업정책지원`으로 바뀌어
  기준축이 될 수 없다(실측: 단위사업 `국방기술개발`은 2016~2027 내내 같은 이름).
- **표기변경(`확정`)**: 같은 단위사업 안에서 `sub_program_key`가 같은데 원문 표기만 다른 경우(`(R&D)`·`(방사청)` 접미, 공백).
  첫 등장 연도 순으로 이웃한 표기끼리 잇되, 연도 범위가 겹치면(같은 기간에 두 표기가 함께 쓰임) `link_basis`에 그렇게 적는다 —
  **순차 개편이 아니라 표기 혼용**이므로 방향(전→후)을 근거로 쓰지 않는다.
- **승계 후보(`후보`)**: `단위사업 = 국방기술개발`(⑤ 탭 대표 축) 안에서 한 세부사업이 Y년에 끝나고 다른 세부사업이 Y+1년에 시작하는 쌍.
  `candidate_count>1`이면 1:1로 확정하지 않는다.
- **신설·종료(`미확인`)**: 국방기술개발 안에서 짝을 찾지 못한 단절.
- **다른 단위사업의 승계는 연결표에 넣지 않는다**: 전 단위사업으로 같은 규칙을 돌리면 후보 쌍이 1,028개(2026-09-19 실측)라 1:1 연결이 불가능하다.
  그 경우 개편 여부는 예산 표의 `sub_program_first_year`/`sub_program_last_year`로만 읽는다(연결 상태 = 미확인).

In [4]:
MINY, MAXY = int(b["fiscal_year"].min()), int(b["fiscal_year"].max())
links = []

# 3-1 표기변경(확정) — 같은 단위사업 안에서 정규화 키가 같은데 원문 표기가 여럿
var = (b.groupby(["unit_program_name", "sub_program_key", "sub_program_name"])
         .agg(fy=("fiscal_year", "min"), ly=("fiscal_year", "max"),
              program_name=("program_name", "last"))
         .reset_index().sort_values(["unit_program_name", "sub_program_key", "fy", "ly"]))
multi = var[var.duplicated(["unit_program_name", "sub_program_key"], keep=False)]
print("표기 변이가 있는 (단위사업, 정규화 키):", multi.groupby(["unit_program_name", "sub_program_key"]).ngroups,
      "| 표기 행:", len(multi))
for (unit_v, skey_v), g in multi.groupby(["unit_program_name", "sub_program_key"]):
    g = g.sort_values(["fy", "ly"]).reset_index(drop=True)
    for i in range(len(g) - 1):
        a, c = g.loc[i], g.loc[i + 1]
        overlap = int(c["fy"]) <= int(a["ly"])
        links.append(dict(program_name=c["program_name"], unit_program_name=unit_v,
                          from_sub_program_name=a["sub_program_name"], from_last_year=int(a["ly"]),
                          to_sub_program_name=c["sub_program_name"], to_first_year=int(c["fy"]),
                          link_type="표기변경", link_status="확정", candidate_count=1,
                          link_basis=("정규화 키 동일('%s') — 공백·(R&D)·(방사청) 표기 차이. %s"
                                      % (skey_v,
                                         ("연도 범위가 겹침(%d~%d vs %d~%d) = 같은 기간 표기 혼용, 전후 관계 아님"
                                          % (int(a["fy"]), int(a["ly"]), int(c["fy"]), int(c["ly"])))
                                         if overlap else
                                         ("%d년까지 → %d년부터" % (int(a["ly"]), int(c["fy"])))))))

# 3-2 국방기술개발 단위사업 안 승계 후보 / 신설·종료 (프로그램명은 2018년에 바뀌므로 단위사업으로만 묶는다)
g0 = b[b["is_unit_tech_dev"] == 1]
kk = (g0.groupby(["unit_program_name", "sub_program_key"])
        .agg(fy=("fiscal_year", "min"), ly=("fiscal_year", "max"), name_last=("sub_program_name", "last"),
             name_first=("sub_program_name", "first"), program_name=("program_name", "last")).reset_index())
print("국방기술개발 세부사업(정규화 키):", len(kk),
      "| 종료(<%d):" % MAXY, int((kk["ly"] < MAXY).sum()), "| 신설(>%d):" % MINY, int((kk["fy"] > MINY).sum()))
matched_to = set()
for _, a in kk[kk["ly"] < MAXY].iterrows():
    cands = kk[kk["fy"] == a["ly"] + 1]
    if len(cands) == 0:
        links.append(dict(program_name=a["program_name"], unit_program_name=a["unit_program_name"],
                          from_sub_program_name=a["name_last"], from_last_year=int(a["ly"]),
                          to_sub_program_name=None, to_first_year=None,
                          link_type="종료", link_status="미확인", candidate_count=0,
                          link_basis=f"{int(a['ly'])}년 이후 사라짐. 같은 단위사업에 {int(a['ly'])+1}년 시작 세부사업 없음"))
        continue
    for _, c in cands.iterrows():
        matched_to.add(c["sub_program_key"])
        links.append(dict(program_name=a["program_name"], unit_program_name=a["unit_program_name"],
                          from_sub_program_name=a["name_last"], from_last_year=int(a["ly"]),
                          to_sub_program_name=c["name_first"], to_first_year=int(c["fy"]),
                          link_type="승계 후보", link_status="후보", candidate_count=int(len(cands)),
                          link_basis=("같은 프로그램·단위사업에서 연도 인접(%d 종료 → %d 시작). 공식 개편 근거 없음 — 후보 %d개"
                                      % (int(a["ly"]), int(c["fy"]), len(cands)))))
for _, c in kk[kk["fy"] > MINY].iterrows():
    if c["sub_program_key"] in matched_to:
        continue
    links.append(dict(program_name=c["program_name"], unit_program_name=c["unit_program_name"],
                      from_sub_program_name=None, from_last_year=None,
                      to_sub_program_name=c["name_first"], to_first_year=int(c["fy"]),
                      link_type="신설", link_status="미확인", candidate_count=0,
                      link_basis=f"{int(c['fy'])}년 처음 등장. 같은 단위사업에 {int(c['fy'])-1}년 종료 세부사업 없음"))

lk = pd.DataFrame(links)
lk["from_last_year"] = lk["from_last_year"].astype("Int64")
lk["to_first_year"] = lk["to_first_year"].astype("Int64")
lk["created_at"] = NOW
lk["created_by"] = USER
print("연결표 행:", len(lk), "|", lk.groupby(["link_type", "link_status"]).size().to_dict())
for r in lk.sort_values(["link_type", "from_last_year"]).itertuples():
    print(f"  [{r.link_type}/{r.link_status}] {r.from_sub_program_name}(~{r.from_last_year}) → "
          f"{r.to_sub_program_name}({r.to_first_year}~) · 후보 {r.candidate_count}")

표기 변이가 있는 (단위사업, 정규화 키): 9 | 표기 행: 18
국방기술개발 세부사업(정규화 키): 18 | 종료(<2027): 3 | 신설(>2016): 13
연결표 행: 22 | {('승계 후보', '후보'): 5, ('신설', '미확인'): 8, ('표기변경', '확정'): 9}
  [승계 후보/후보] 핵심부품·SW개발/신개념기술시범(R&D)(~2020) → 부품국산화 개발지원(R&D)(2021~) · 후보 1
  [승계 후보/후보] 핵심기술개발(R&D)(~2022) → 개별핵심기술(R&D)(2023~) · 후보 3
  [승계 후보/후보] 핵심기술개발(R&D)(~2022) → 전용기술(R&D)-II(2023~) · 후보 3
  [승계 후보/후보] 핵심기술개발(R&D)(~2022) → 패키지 핵심기술(R&D)(2023~) · 후보 3
  [승계 후보/후보] 신속연구개발(R&D)(~2024) → 신속시범(R&D)(2025~) · 후보 1
  [신설/미확인] nan(~<NA>) → 국방AI선도 연구협력(R&D)(2026~) · 후보 0
  [신설/미확인] nan(~<NA>) → 국방반도체(R&D)(2027~) · 후보 0
  [신설/미확인] nan(~<NA>) → 국방 양자암호통신·복합센서 기술개발(R&D)(2026~) · 후보 0
  [신설/미확인] nan(~<NA>) → 미래국방가교기술개발(R&D)(2024~) · 후보 0
  [신설/미확인] nan(~<NA>) → 미래도전국방기술(R&D)(2022~) · 후보 0
  [신설/미확인] nan(~<NA>) → 북극항로 해양정보 예보 및 음향탐지성능분석 시스템 개발(R&D)(2027~) · 후보 0
  [신설/미확인] nan(~<NA>) → 신속연구개발(R&D)(2022~) · 후보 0
  [신설/미확인] nan(~<NA>) → 유무인복합체계용첨단항공추진체계(R&D)(2027~) · 후보 0
  [표기변경/확정] 기술/연구지원(~2016) → 기술/연구지원(R&D)(2017~) · 후보 1
  [표기변경/확

## 4. ⑤ 탭 예산 3선 — `budget_group_candidate` ↔ `v_budget_rnd_yearly` 대조

`v_budget_rnd_yearly`(`db/schema.sql`)는 raw 를 직접 집계하며 `unit_program_name='국방기술개발'`,
`sub_program_name LIKE '부품국산화%'`, `LIKE '국방반도체%'`(접두)를 쓴다.
이 노트북의 후보 열은 `반도체`·`부품국산화` **포함**(접두 아님)이라 더 넓다 — 차이가 있으면 아래에서 드러난다.

In [5]:
mine = (b.groupby("fiscal_year")
          .apply(lambda g: pd.Series({
              "total_gov_100m": round(g["gov_plan_100m_krw"].sum(), 1),
              "tech_dev_gov_100m": round(g.loc[g["is_unit_tech_dev"] == 1, "gov_plan_100m_krw"].sum(), 1),
              "tech_dev_confirmed_100m": round(g.loc[g["is_unit_tech_dev"] == 1, "confirmed_100m_krw"].sum(), 1),
              "localization_gov_100m": round(g.loc[g["budget_group_candidate"] == "부품국산화", "gov_plan_100m_krw"].sum(), 1),
              "semiconductor_gov_100m": round(g.loc[g["budget_group_candidate"] == "국방반도체", "gov_plan_100m_krw"].sum(), 1),
              "amount_basis": g["amount_basis"].iloc[0]}), include_groups=False)
          .reset_index())
view = q("SELECT CAST(fiscal_year AS UNSIGNED) fiscal_year, amount_basis, total_gov_100m, tech_dev_gov_100m, "
         "tech_dev_confirmed_100m, localization_gov_100m, semiconductor_gov_100m, row_count FROM v_budget_rnd_yearly ORDER BY 1")
cmpdf = mine.merge(view, on="fiscal_year", suffixes=("_nb", "_view"))
for c in ("total_gov_100m", "tech_dev_gov_100m", "tech_dev_confirmed_100m", "localization_gov_100m", "semiconductor_gov_100m"):
    cmpdf[c + "_diff"] = cmpdf[c + "_nb"].astype(float) - cmpdf[c + "_view"].astype(float)
print("연도별 정부안 합(억원) — 노트북 vs v_budget_rnd_yearly")
print(cmpdf[["fiscal_year", "amount_basis_nb", "amount_basis_view", "total_gov_100m_nb", "total_gov_100m_view",
             "tech_dev_gov_100m_nb", "tech_dev_gov_100m_view", "localization_gov_100m_nb", "semiconductor_gov_100m_nb"]].to_string(index=False))
diff_cols = [c for c in cmpdf.columns if c.endswith("_diff")]
print("차이 합계:", {c: float(cmpdf[c].abs().sum()) for c in diff_cols})
print("amount_basis 불일치 연도:", cmpdf.loc[cmpdf["amount_basis_nb"] != cmpdf["amount_basis_view"], "fiscal_year"].tolist())
print("국방반도체 후보가 잡힌 연도:", sorted(b.loc[b['budget_group_candidate'] == '국방반도체', 'fiscal_year'].unique().tolist()),
      "| 부품국산화:", sorted(b.loc[b['budget_group_candidate'] == '부품국산화', 'fiscal_year'].unique().tolist()))

연도별 정부안 합(억원) — 노트북 vs v_budget_rnd_yearly


 fiscal_year amount_basis_nb amount_basis_view total_gov_100m_nb total_gov_100m_view tech_dev_gov_100m_nb tech_dev_gov_100m_view localization_gov_100m_nb semiconductor_gov_100m_nb
        2016              확정                확정          116803.3            116803.3               8163.5                 8163.5                        0                         0
        2017              확정                확정          121589.5            121589.5               8135.8                 8135.8                        0                         0
        2018              확정                확정          134825.5            134825.5               9107.8                 9107.8                        0                         0
        2019              확정                확정          153733.2            153733.2               9487.8                 9487.8                        0                         0
        2020              확정                확정          166915.4            166915.4              10

## 5. 적재 (대상이 비어 있을 때만) + `clean_excluded_row` + `meta_load_log`

In [6]:
if count("clean_krit_task") > 0:
    print("clean_krit_task 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_krit_task", raw("raw_krit_task"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["round_id", "notice_type", "task_no", "round_year", "round_seq", "program_type", "task_name",
            "gov_fund_100m_krw", "dev_period_months", "is_counted",
            "raw_row_id", "source_file", "cleaned_at", "task_seq_text", "gov_fund_text", "gov_fund_unit_text",
            "dev_period_text", "note", "notice_order", "is_latest", "dup_task_row_count", "cleaned_by"]
    n_ins = insert_df("clean_krit_task", k, cols)
    log_stage(cur, "krit_task", "clean_krit_task", "중복 처리 후",
              "raw 1:1(차수×공고유형×과제번호 재부여 — 원문 순번은 표별로 1부터 다시 시작해 PK 불가)", n_ins,
              "제외 0. 원문 순번을 그대로 쓰면 %d행이 충돌한다(task_seq_text 에 보존)" % int(dup_orig),
              f"{NB_TAG} §1", USER)
    log_stage(cur, "krit_task", "clean_krit_task", "관련 후보",
              "같은 차수·같은 과제명 중 최신 공고 1건(is_latest=1)", int(k["is_latest"].sum()),
              "예비→본공고(26-2차 20쌍)·수정→재공고(25-1차 3쌍)는 같은 과제의 재공고라 합산 금지. 공고 순서 예비<본공고<수정<재공고는 잠정 규칙",
              f"{NB_TAG} §1", USER)
    conn.commit(); print("clean_krit_task 적재:", count("clean_krit_task"))

if count("clean_openfiscal_program_budget") > 0:
    print("clean_openfiscal_program_budget 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_openfiscal_program_budget", raw("raw_openfiscal_program_budget"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["raw_row_id", "fiscal_year", "program_name", "unit_program_name", "sub_program_name", "sub_program_key",
            "sub_program_first_year", "sub_program_last_year",  # 09-21 expense_type·outlay_type 제외(97.5%·99.6% 단일, 금액 0.6%) — raw 보존
            "gov_plan_krw_k", "gov_plan_100m_krw", "confirmed_krw_k", "confirmed_100m_krw",
            "amount_basis", "is_unconfirmed", "is_unit_tech_dev", "budget_group_candidate", "budget_group_basis",
            "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_openfiscal_program_budget", b, cols)
    log_stage(cur, "openfiscal_program_budget", "clean_openfiscal_program_budget", "중복 처리 후",
              "raw 1:1(회계연도×프로그램×단위사업×세부사업 고유)", n_ins,
              "제외 0. 계정명(전부 공란)·소관명·회계명·분야명·부문명·No. 는 열 자체를 제외. 2027 %d행은 국회확정 전(is_unconfirmed=1, 0원 아님)"
              % int(b["is_unconfirmed"].sum()),
              f"{NB_TAG} §2", USER)
    log_stage(cur, "openfiscal_program_budget", "clean_openfiscal_program_budget", "관련 후보",
              "⑤ 탭 예산 3선 키워드 후보(budget_group_candidate NOT NULL)", int(b["budget_group_candidate"].notna().sum()),
              "국방기술개발 %d · 부품국산화 %d · 국방반도체 %d. 뒤 둘은 국방기술개발 단위사업 안에도 있어 합산 시 중복"
              % (int((b["budget_group_candidate"] == "국방기술개발").sum()),
                 int((b["budget_group_candidate"] == "부품국산화").sum()),
                 int((b["budget_group_candidate"] == "국방반도체").sum())),
              f"{NB_TAG} §2", USER)
    conn.commit(); print("clean_openfiscal_program_budget 적재:", count("clean_openfiscal_program_budget"))

if count("clean_openfiscal_program_link") > 0:
    print("clean_openfiscal_program_link 비어 있지 않음 → 건너뜀")
else:
    cols = ["program_name", "unit_program_name", "from_sub_program_name", "from_last_year", "to_sub_program_name",
            "to_first_year", "link_type", "link_status", "candidate_count", "link_basis", "created_at", "created_by"]
    n_ins = insert_df("clean_openfiscal_program_link", lk, cols)
    vc = lk["link_status"].value_counts().to_dict()
    log_stage(cur, "openfiscal_program_budget", "clean_openfiscal_program_link", "관련 후보",
              "세부사업명 개편 연결(확정=표기 차이만, 승계는 후보)", n_ins,
              "확정 %d · 후보 %d · 미확인 %d. 국방기술개발 단위사업 + 표기변경만 담았다 — 전 단위사업으로 넓히면 후보 쌍 1,028개라 1:1 연결 불가"
              % (vc.get("확정", 0), vc.get("후보", 0), vc.get("미확인", 0)),
              f"{NB_TAG} §3", USER)
    conn.commit(); print("clean_openfiscal_program_link 적재:", count("clean_openfiscal_program_link"))

ex = pd.DataFrame(excluded, columns=["table_name", "raw_row_id", "reason_code", "note"])
if len(ex):
    cur.execute("SELECT table_name, raw_row_id FROM clean_excluded_row")
    have = set(cur.fetchall())
    ex = ex[~ex.apply(lambda r: (r["table_name"], r["raw_row_id"]) in have, axis=1)]
    if len(ex):
        ex["excluded_by"] = USER
        insert_df("clean_excluded_row", ex, ["table_name", "raw_row_id", "reason_code", "note", "excluded_by"])
        conn.commit()
print("이번 노트북 제외 행:", len(ex), "| clean_excluded_row 현재:", count("clean_excluded_row"))

clean_krit_task 비어 있지 않음 → 건너뜀
clean_openfiscal_program_budget 비어 있지 않음 → 건너뜀
clean_openfiscal_program_link 비어 있지 않음 → 건너뜀
이번 노트북 제외 행: 0 | clean_excluded_row 현재: 17


## 6. 적재 후 검증 — 검산 `raw = clean + excluded`, 키 규칙, 연도별 합계

In [7]:
_pairs = [("raw_krit_task", "clean_krit_task"), ("raw_openfiscal_program_budget", "clean_openfiscal_program_budget")]
# 원본 = 파일(read_raw), 정제·제외 = DB. raw = clean + excluded 검산
chk = pd.DataFrame({"tbl": [r for r, _ in _pairs], "raw_n": [len(raw(r)) for r, _ in _pairs], "clean_n": [count(c) for _, c in _pairs]})
_ex = q("SELECT table_name tbl, COUNT(*) excl_n FROM clean_excluded_row GROUP BY table_name")
chk = chk.merge(_ex, on="tbl", how="left").fillna({"excl_n": 0})
chk["excl_n"] = chk["excl_n"].astype(int); chk["gap"] = chk["raw_n"] - chk["clean_n"] - chk["excl_n"]
display(chk)
display(q("SELECT round_id, round_year, COUNT(*) rows_, SUM(is_latest) latest_n, SUM(is_counted) counted_n, "
          "COUNT(DISTINCT notice_type) notice_types, ROUND(SUM(CASE WHEN is_latest=1 THEN gov_fund_100m_krw END),2) fund_100m "
          "FROM clean_krit_task GROUP BY 1,2 ORDER BY 1"))
display(q("SELECT notice_type, COUNT(*) rows_, SUM(is_latest) latest_n FROM clean_krit_task GROUP BY 1 ORDER BY 2 DESC"))
display(q("SELECT round_id, task_name, COUNT(*) n FROM clean_krit_task GROUP BY 1,2 HAVING SUM(is_latest) <> 1"))   # 0행 기대
display(q("SELECT program_type, COUNT(*) n, SUM(is_latest) latest_n FROM clean_krit_task GROUP BY 1"))
display(q("SELECT gov_fund_unit_text, COUNT(*) n, SUM(gov_fund_100m_krw IS NULL) null_n FROM clean_krit_task GROUP BY 1"))

display(q("SELECT fiscal_year, amount_basis, MAX(is_unconfirmed) unconf, COUNT(*) rows_, "
          "ROUND(SUM(gov_plan_100m_krw),1) gov_100m, ROUND(SUM(confirmed_100m_krw),1) confirmed_100m, "
          "ROUND(SUM(CASE WHEN is_unit_tech_dev=1 THEN gov_plan_100m_krw END),1) tech_dev_100m, "
          "ROUND(SUM(CASE WHEN budget_group_candidate='부품국산화' THEN gov_plan_100m_krw END),1) local_100m, "
          "ROUND(SUM(CASE WHEN budget_group_candidate='국방반도체' THEN gov_plan_100m_krw END),1) semi_100m "
          "FROM clean_openfiscal_program_budget GROUP BY 1,2 ORDER BY 1"))
display(q("SELECT fiscal_year, total_gov_100m, tech_dev_gov_100m, localization_gov_100m, semiconductor_gov_100m, amount_basis "
          "FROM v_budget_rnd_yearly ORDER BY 1"))
display(q("SELECT link_type, link_status, COUNT(*) n, SUM(candidate_count>1) multi FROM clean_openfiscal_program_link GROUP BY 1,2"))
display(q("SELECT from_sub_program_name, from_last_year, to_sub_program_name, to_first_year, link_type, link_status, candidate_count "
          "FROM clean_openfiscal_program_link WHERE link_type <> '신설' ORDER BY from_last_year, to_first_year"))
display(q("SELECT log_id, dataset_key, table_name, stage, row_count FROM meta_load_log WHERE log_id > 92 ORDER BY log_id"))

,tbl,raw_n,clean_n,excl_n,gap
0,raw_krit_task,96,96,0,0
1,raw_openfiscal_program_budget,2860,2860,0,0


,round_id,round_year,rows_,latest_n,counted_n,notice_types,fund_100m
0,23-4,2023,18,18,18,1,762.19
1,24-1,2024,11,11,11,1,None
2,25-1,2025,25,22,22,2,1344.60
3,26-1,2026,2,2,2,1,53.78
4,26-2,2026,40,20,20,2,1571.50


,notice_type,rows_,latest_n
0,본공고,40,40
1,예비,31,11
2,수정,22,19
3,재공고,3,3


,round_id,task_name,n


,program_type,n,latest_n
0,핵심부품,72,57
1,수출연계,3,3
2,전략부품,19,11
3,NaN,2,2


...

,fiscal_year,amount_basis,unconf,rows_,gov_100m,confirmed_100m,tech_dev_100m,local_100m,semi_100m
0,2016,확정,0,229,116803.3,116398.1,8163.5,None,None
1,2017,확정,0,216,121589.5,121969.5,8135.8,None,None
2,2018,확정,0,219,134825.5,135202.9,9107.8,None,None
3,2019,확정,0,215,153733.2,153733.2,9487.8,None,None
4,2020,확정,0,223,166915.4,166804.4,10053.3,None,None
5,2021,확정,0,233,170737.8,169964.2,13877.9,886.4,None
6,2022,확정,0,248,173364.8,166917.3,21739.6,1691.1,None
7,2023,확정,0,241,170179.0,169168.9,27174.6,1845.1,None
8,2024,확정,0,251,177985.7,176532.4,23738.7,1232.7,None
9,2025,확정,0,260,180712.5,178461.6,25371.1,1278.8,None


,fiscal_year,total_gov_100m,tech_dev_gov_100m,localization_gov_100m,semiconductor_gov_100m,amount_basis
0,2016,116803.3,8163.5,0.0,0.0,확정
1,2017,121589.5,8135.8,0.0,0.0,확정
2,2018,134825.5,9107.8,0.0,0.0,확정
3,2019,153733.2,9487.8,0.0,0.0,확정
4,2020,166915.4,10053.3,0.0,0.0,확정
5,2021,170737.8,13877.9,886.4,0.0,확정
6,2022,173364.8,21739.6,1691.1,0.0,확정
7,2023,170179.0,27174.6,1845.1,0.0,확정
8,2024,177985.7,23738.7,1232.7,0.0,확정
9,2025,180712.5,25371.1,1278.8,0.0,확정


,link_type,link_status,n,multi
0,표기변경,확정,9,0
1,승계 후보,후보,5,3
2,신설,미확인,8,0


,from_sub_program_name,from_last_year,to_sub_program_name,to_first_year,link_type,link_status,candidate_count
0,기술/연구지원,2016,기술/연구지원(R&D),2017,표기변경,확정,1
1,핵심부품·SW개발/신개념기술시범,2016,핵심부품·SW개발/신개념기술시범(R&D),2017,표기변경,확정,1
2,해군전술C4I체계 성능개량,2016,해군전술C4I체계성능개량(R&D),2017,표기변경,확정,1
3,민군기술협력(R&D),2017,민군기술협력(R&D)(방사청),2018,표기변경,확정,1
4,군사정보통합처리체계(MIMS)성능개량(R&D),2019,군사정보통합처리체계(MIMS)성능개량,2020,표기변경,확정,1
5,지상전술C4I체계확장(R&D),2019,지상전술C4I체계확장,2020,표기변경,확정,1
6,해군전술C4I체계성능개량(R&D),2019,해군전술C4I체계성능개량,2020,표기변경,확정,1
7,KDX-I성능개량(R&D),2019,KDX-I성능개량,2020,표기변경,확정,1
8,핵심부품·SW개발/신개념기술시범(R&D),2020,부품국산화 개발지원(R&D),2021,승계 후보,후보,1
9,연합군사정보처리체계(MIMS-C) 성능개량(R&D),2021,연합군사정보처리체계(MIMS-C) 성능개량,2022,표기변경,확정,1


,log_id,dataset_key,table_name,stage,row_count
0,98,krit_task,clean_krit_task,중복 처리 후,96
1,99,krit_task,clean_krit_task,관련 후보,73
2,100,openfiscal_program_budget,clean_openfiscal_program_budget,중복 처리 후,2860
3,101,openfiscal_program_budget,clean_openfiscal_program_budget,관련 후보,93
4,103,dapa_overseas_plan_api,clean_dapa_overseas_plan_api,중복 처리 후,13615
...,...,...,...,...,...
56,155,dapa_bid_notice,clean_dapa_bid_notice,중복 처리 후,10840
57,156,dapa_contract,clean_dapa_contract,중복 처리 후,43105
58,157,dapa_overseas_bid_result,clean_dapa_overseas_bid_result,중복 처리 후,2494
59,158,dapa_domestic_plan,clean_dapa_domestic_plan,중복 처리 후,35859


## 남은 일 (이 노트북이 하지 않은 것)

- `clean_krit_task`에는 HS6 대응 열이 없다(2026-09-21 카테고리 맵 폐기, `v_review_list` B1 열도 제거). KRIT 과제는 HS6 품목군과 잇지 않는다.
- `notice_order`(예비<본공고<수정<재공고)는 **잠정**이다. 25-1차는 `수정`(22건) 뒤 `재공고`(3건)가 나온 것으로 보았다 — 팀 확인 필요.
- 24-1차 예비 11건의 `총과제비(억원)`는 정부지원금과 다른 지표라 `gov_fund_100m_krw`에 넣지 않았다(`extra_json`에만 있음).
  차수별 금액을 비교할 때 이 11건은 금액 NULL이다.
- `clean_openfiscal_program_link`는 국방기술개발 단위사업 + 표기변경만 담았다. 나머지 단위사업의 개편 구간은 **미확인**이며,
  필요하면 열린재정 「사업 연혁」이나 예산안 개요에서 공식 근거를 찾아 `link_status`를 올려야 한다.
- `budget_group_candidate`는 키워드 후보다. ⑤ 탭에 쓰려면 세부사업 목록을 팀이 확정하고 `budget_group_basis`에 근거를 남겨야 한다.
- 기존 뷰 `v_budget_rnd_yearly`는 raw 직접 집계 그대로 두었다. clean 기준으로 바꿀지는 `schema-design.md` §7-20 ④와 함께 결정.